In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm

In [2]:
def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

In [3]:
cleaned_labels = pd.read_excel("../../data/TCGA/clinical/TCGA-CDR-SupplementalTableS1.xlsx")

In [4]:
OS_labels = cleaned_labels[["bcr_patient_barcode", "OS", "OS.time"]]

In [5]:
PFS_labels = pd.read_excel("../../data/TCGA/clinical/TCGA-CDR-SupplementalTableS1.xlsx", sheet_name='ExtraEndpoints')

In [6]:
PFS_labels = PFS_labels[["bcr_patient_barcode", "PFS", "PFS.time"]]

In [7]:
# Merge the two dataframes on the 'bcr_patient_barcode' column
survival_labels = pd.merge(OS_labels, PFS_labels, on='bcr_patient_barcode', how='outer')

In [8]:
rename_cols = {
    "OS": "OS_event",
    "OS.time": "OS_time",
    "PFS": "PFS_event",
    "PFS.time": "PFS_time"
}

In [9]:
survival_labels.rename(columns=rename_cols, inplace=True)

In [10]:
survival_labels.head()

,bcr_patient_barcode,OS_event,OS_time,PFS_event,PFS_time
0,TCGA-02-0001,1.0,358.0,1.0,137.0
1,TCGA-02-0003,1.0,144.0,1.0,40.0
2,TCGA-02-0004,1.0,345.0,1.0,314.0
3,TCGA-02-0006,1.0,558.0,1.0,302.0
4,TCGA-02-0007,1.0,705.0,1.0,518.0


In [11]:
survival_labels.to_csv("../../data/TCGA/clinical/cleaned_survival_labels.csv", index=False)

In [12]:
survival_labels = pd.read_csv("../../data/TCGA/clinical/cleaned_survival_labels.csv")

In [14]:
survival_labels.head()

,bcr_patient_barcode,OS_event,OS_time,PFS_event,PFS_time
0,TCGA-02-0001,1.0,358.0,1.0,137.0
1,TCGA-02-0003,1.0,144.0,1.0,40.0
2,TCGA-02-0004,1.0,345.0,1.0,314.0
3,TCGA-02-0006,1.0,558.0,1.0,302.0
4,TCGA-02-0007,1.0,705.0,1.0,518.0


In [15]:
embeddings = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/RNA_embeddings/TCGA_processed_HVG.pkl")

In [16]:
embeddings['TCGA-AB-2911']['BulkFormer_HVG_1250']

array([-3.24506253e-01, -2.60605145e+00, -5.70197761e-01, -8.76725316e-01,
        8.71399403e-01,  1.71529666e-01, -7.38718629e-01,  1.31308794e+00,
       -1.88633716e+00, -1.28142309e+00,  2.04639986e-01, -2.11778972e-02,
       -2.13997394e-01,  3.93419683e-01, -8.71468782e-02,  3.68212879e-01,
        1.31070602e+00, -7.21798092e-02, -1.01183081e+00,  1.50637671e-01,
        2.02884361e-01, -8.27679813e-01,  1.33338356e+00,  4.59863037e-01,
       -7.81250358e-01,  9.27442670e-01, -5.58052957e-01, -1.59003091e+00,
       -9.92460772e-02, -4.58003521e-01, -8.01043332e-01,  5.17141342e-01,
        3.88639160e-02,  3.43681037e-01,  2.47906402e-01,  3.81626368e-01,
       -7.61792839e-01, -1.44309604e+00, -5.00080490e+00, -5.90622485e-01,
       -5.67808092e-01,  2.87845910e-01, -1.12099993e+00, -2.28620574e-01,
        9.74283218e-01,  1.57670677e-01,  7.78354406e-01, -1.78540777e-02,
       -4.01572764e-01, -8.95212770e-01, -1.00125723e-01, -3.31693725e-03,
       -3.64665449e-01, -

In [17]:
patients_with_cleaned_labels = set(survival_labels['bcr_patient_barcode'])
len(patients_with_cleaned_labels)

11160

In [18]:
patients_in_embeddings_with_PFS_labels = set()
for patient_id in embeddings.keys():
    if not any(pd.isna([embeddings[patient_id]['PFS_time'], embeddings[patient_id]['PFS_event']])):
        patients_in_embeddings_with_PFS_labels.add(patient_id)

print(f"Out of {len(embeddings)} patients in the embeddings, {len(patients_in_embeddings_with_PFS_labels)} have PFS labels.")

Out of 10478 patients in the embeddings, 10388 have PFS labels.


In [19]:
patients_in_embeddings_with_cleaned_labels = [patient for patient in patients_in_embeddings_with_PFS_labels if patient in patients_with_cleaned_labels]
len(patients_in_embeddings_with_cleaned_labels)

10388

In [20]:
# Create copy of embeddings with new labels for PFS_time and PFS_event
nan_patients = set()
new_embeddings = {}
for patient in tqdm(patients_in_embeddings_with_cleaned_labels):
    new_embeddings[patient] = embeddings[patient].copy()
    # Get the corresponding PFS_time and PFS_event from survival_labels
    pfs_time = survival_labels[survival_labels['bcr_patient_barcode'] == patient]['PFS_time'].values[0]
    pfs_event = survival_labels[survival_labels['bcr_patient_barcode'] == patient]['PFS_event'].values[0]
    os_time = survival_labels[survival_labels['bcr_patient_barcode'] == patient]['OS_time'].values[0]
    os_event = survival_labels[survival_labels['bcr_patient_barcode'] == patient]['OS_event'].values[0]
    # Update the embedding with the new labels only if the PFS label's values are not NaN
    if not any(pd.isna([pfs_time, pfs_event])):
        new_embeddings[patient]['PFS_time'] = pfs_time
        new_embeddings[patient]['PFS_event'] = pfs_event
        new_embeddings[patient]['OS_time'] = os_time
        new_embeddings[patient]['OS_event'] = os_event
    else:
        nan_patients.add(patient)

  0%|          | 0/10388 [00:00<?, ?it/s]

In [21]:
len(nan_patients)

149

In [22]:
# Ensure no NaN values in the new embeddings' PFS values
for patient in new_embeddings:
    if any(pd.isna([new_embeddings[patient]['PFS_time'], new_embeddings[patient]['PFS_event']])):
        print(f"NaN values found for patient {patient}")

In [23]:
len(new_embeddings)

10388

In [24]:
# Compute number of patients with differening PFS_time, PFS_event between the original and new embeddings
differing_patients = set()
for patient in new_embeddings:
    if (new_embeddings[patient]['PFS_time'] != embeddings[patient]['PFS_time']) or \
       (new_embeddings[patient]['PFS_event'] != embeddings[patient]['PFS_event']):
        differing_patients.add(patient)
len(differing_patients)

290

In [25]:
print(f"Number of patients with NaN cleaned values in PFS_time or PFS_event: {len(nan_patients)}")

Number of patients with NaN cleaned values in PFS_time or PFS_event: 149


In [26]:
pickle_data(new_embeddings, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/RNA_embeddings/TCGA_processed_HVG_cleaned.pkl")